# DAPO

$$
\mathcal{J}_{\text{DAPO}}(\theta) = \mathbb{E}_{q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}(O \mid q)} \Bigg[ \frac{1}{\sum_{i=1}^G |o_i|} \sum_{i=1}^G \sum_{t=1}^{|o_i|} \min \Bigg( r_{i,t}(\theta) \hat{A}_{i,t}, \text{clip} \left( r_{i,t}(\theta), 1 - \epsilon_{\text{low}}, 1 + \epsilon_{\text{high}} \right) \hat{A}_{i,t} \Bigg) \Bigg]
$$

$$
r_{i,t}(\theta)=\frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})}, \quad \hat{A}_{i,t}=\frac{r_i-\text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}
$$

In [ ]:
import torch

def dapo_loss(rewards, logp_per_token, old_logp_per_token, completion_mask=None, clip_epsilon_low=0.2, clip_epsilon_high=0.28):
    """
    Args:
        rewards (torch.Tensor): 奖励, shape: [batch_size, num_generation]
        logp_per_token (torch.Tensor): 策略模型logp, shape: [batch_size, num_generation, seq_len]
        old_logp_per_token (torch.Tensor): 旧策略模型logp, shape: [batch_size, num_generation, seq_len]
        completion_mask (torch.Tensor | None): 有效token掩码, shape: [batch_size, num_generation, seq_len]
        clip_epsilon_low (float): 下界裁剪参数
        clip_epsilon_high (float): 上界裁剪参数
    """
    if completion_mask is None:
        completion_mask = torch.ones_like(logp_per_token)

    mean_grouped_rewards = rewards.mean(dim=-1, keepdim=True) # shape: [batch_size, 1]
    std_grouped_rewards = rewards.std(dim=-1, keepdim=True)   # shape: [batch_size, 1]

    advantage_per_sequence = (rewards - mean_grouped_rewards) / (std_grouped_rewards + 1e-8) # [batch_size, num_generation]
    advantage_per_token = advantage_per_sequence.unsqueeze(-1).expand_as(logp_per_token) # [batch_size, num_generation, seq_len]

    importance_ratio = torch.exp(logp_per_token - old_logp_per_token)
    clipped_importance_ratio = torch.clamp(importance_ratio, 1.0 - clip_epsilon_low, 1.0 + clip_epsilon_high)

    adv1 = importance_ratio * advantage_per_token
    adv2 = clipped_importance_ratio * advantage_per_token
    policy_objective_per_token = torch.min(adv1, adv2) # [batch_size, num_generation, seq_len]

    masked_policy_objective = policy_objective_per_token * completion_mask
    valid_token_count = completion_mask.sum().clamp_min(1.0)

    loss = -masked_policy_objective.sum() / valid_token_count
    return loss
